In [ ]:
import pandas as pd
import numpy as np

# 1. CARGA DE DATOS
customers = pd.read_csv('premium_customers.csv', dtype=str)
txns = pd.read_csv('premium_transactions.csv', dtype=str)
products = pd.read_csv('premium_products.csv', dtype=str)
stores = pd.read_csv('premium_stores.csv', dtype=str)

orig_cust_count = len(customers)
orig_txns_count = len(txns)

# ==========================================
# 2. LIMPIEZA DE CLIENTES (premium_customers.csv)
# ==========================================
customers['email'] = customers['email'].fillna('').str.strip().str.lower().replace('', np.nan)
customers['phone'] = customers['phone'].fillna('').str.replace(r'\D', '', regex=True)
customers['phone'] = np.where(customers['phone'].str.len() >= 10, customers['phone'].str[-10:], np.nan)
customers['name'] = customers['name'].fillna('').str.split().str.join(' ').str.title().replace('', np.nan)

valid_cities = {
    'buenos aires': 'Buenos Aires', 'caba': 'Buenos Aires', 'bs as': 'Buenos Aires', 'bs. as.': 'Buenos Aires',
    'cordoba': 'Córdoba', 'córdoba': 'Córdoba',
    'rosario': 'Rosario',
    'mendoza': 'Mendoza'
}
customers['city'] = customers['city'].fillna('').str.strip().str.lower().map(valid_cities)

tier_map = {
    'bronze': 'Bronze', 'bronce': 'Bronze',
    'silver': 'Silver', 'plata': 'Silver',
    'gold': 'Gold', 'oro': 'Gold'
}
customers['loyalty_tier'] = customers['loyalty_tier'].fillna('').str.strip().str.lower().map(tier_map)

# Deduplicación transitiva (mismo email O mismo teléfono)
parent = list(range(len(customers)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(i, j):
    ri, rj = find(i), find(j)
    if ri != rj:
        parent[max(ri, rj)] = min(ri, rj)

seen_email, seen_phone = {}, {}
for idx, row in customers.iterrows():
    if pd.notna(row['email']):
        if row['email'] in seen_email:
            union(idx, seen_email[row['email']])
        else:
            seen_email[row['email']] = idx
    if pd.notna(row['phone']):
        if row['phone'] in seen_phone:
            union(idx, seen_phone[row['phone']])
        else:
            seen_phone[row['phone']] = idx

customers['group_id'] = [find(i) for i in range(len(customers))]
customers_clean = customers.groupby('group_id', as_index=False).first()
cust_dups_removed = orig_cust_count - len(customers_clean)

customers_clean = customers_clean.sort_values(by=['email', 'name'], na_position='last').reset_index(drop=True)
customers_clean['customer_id'] = [f'C{i+1:04d}' for i in range(len(customers_clean))]
customers_clean = customers_clean[['customer_id', 'name', 'email', 'phone', 'city', 'loyalty_tier']]

# ==========================================
# 3. LIMPIEZA DE TRANSACCIONES (premium_transactions.csv)
# ==========================================
txns['txn_id'] = txns['txn_id'].fillna('').str.strip().str.upper()
txns['ingested_dt'] = pd.to_datetime(txns['ingested_at'], errors='coerce', utc=True)
txns = txns.sort_values('ingested_dt', ascending=False).drop_duplicates(subset=['txn_id'], keep='first').copy()
txn_dups_removed = orig_txns_count - len(txns)

raw_dt = txns['txn_datetime'].fillna('').str.strip()
is_utc = raw_dt.str.endswith('Z')
dt_utc = pd.to_datetime(raw_dt.where(is_utc), errors='coerce', utc=True).dt.tz_convert('America/Argentina/Buenos_Aires').dt.tz_localize(None)

has_slash = raw_dt.str.contains('/')
dt_local_slash = pd.to_datetime(raw_dt.where(~is_utc & has_slash), format='%d/%m/%Y %H:%M:%S', errors='coerce')
mask_fb = ~is_utc & has_slash & dt_local_slash.isna()
dt_local_slash = dt_local_slash.fillna(pd.to_datetime(raw_dt.where(mask_fb), dayfirst=True, errors='coerce'))
dt_local_std = pd.to_datetime(raw_dt.where(~is_utc & ~has_slash), format='mixed', errors='coerce')

txns['dt_ar'] = dt_utc.fillna(dt_local_slash).fillna(dt_local_std)
txns['txn_datetime'] = txns['dt_ar'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Normalizar store_code -> S01 a S12
store_num = pd.to_numeric(txns['store_code'].fillna('').str.extract(r'(\d+)')[0], errors='coerce')
valid_store_mask = store_num.between(1, 12)
txns['store_code'] = np.where(valid_store_mask, 'S' + store_num.fillna(0).astype(int).astype(str).str.zfill(2), np.nan)

# Normalizar sku -> P-0001 (soporta P0029, p-0032, p 0031)
sku_clean = txns['sku'].fillna('').str.strip().str.upper()
sku_digits = sku_clean.str.extract(r'^P[-\s]?(\d+)$')[0]
txns['sku'] = np.where(sku_digits.notna(), 'P-' + sku_digits.str.zfill(4), sku_clean)
valid_skus = set(products['sku'].str.strip().str.upper())
valid_sku_mask = txns['sku'].isin(valid_skus)

# Normalizar quantity
word_to_num = {'uno': '1', 'dos': '2', 'tres': '3', 'cuatro': '4', 'cinco': '5'}
qty_str = txns['quantity'].fillna('').str.strip().str.lower().replace(word_to_num)
qty_num = pd.to_numeric(qty_str, errors='coerce')
valid_qty_mask = qty_num.notna() & (qty_num % 1 == 0) & (qty_num > 0) & (qty_num <= 100)
txns['quantity'] = qty_num

# Normalizar discount
disc_raw = txns['discount'].fillna('').str.strip()
has_pct = disc_raw.str.endswith('%')
disc_num = pd.to_numeric(disc_raw.str.rstrip('%'), errors='coerce')
disc_val = np.where(disc_raw == '', 0.0, np.where(has_pct | (disc_num > 1), disc_num / 100.0, disc_num))
valid_disc_mask = pd.notna(disc_val) & (disc_val >= 0.0) & (disc_val <= 1.0)
txns['discount'] = np.round(disc_val, 2)

# Validar fecha futura (> 16/11/2026 23:59:59)
cutoff_date = pd.Timestamp('2026-11-16 23:59:59')
future_date_mask = txns['dt_ar'] > cutoff_date

conditions = [~valid_store_mask, ~valid_sku_mask, ~valid_qty_mask, ~valid_disc_mask, future_date_mask]
reasons = ['invalid_store', 'unknown_sku', 'invalid_quantity', 'invalid_discount', 'future_date']
txns['reason'] = np.select(conditions, reasons, default='')

rejected_df = txns[txns['reason'] != ''][['txn_id', 'reason']].copy()
clean_df = txns[txns['reason'] == ''].copy()
clean_df['quantity'] = clean_df['quantity'].astype(int)

def parse_price(s):
    s = s.fillna('').str.strip().str.replace('$', '', regex=False).str.strip()
    has_comma = s.str.contains(',')
    s_ar = s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    return pd.to_numeric(np.where(has_comma, s_ar, s), errors='coerce')

clean_df['unit_price_parsed'] = parse_price(clean_df['unit_price'])
products['sku'] = products['sku'].str.strip().str.upper()
products['list_price_num'] = parse_price(products['list_price'])
clean_df = clean_df.merge(products[['sku', 'product_name', 'category', 'list_price_num']], on='sku', how='left')

use_catalog = clean_df['unit_price_parsed'].isna()
clean_df['price_source'] = np.where(use_catalog, 'catalog', 'transaction')
clean_df['unit_price'] = np.round(np.where(use_catalog, clean_df['list_price_num'], clean_df['unit_price_parsed']), 2)

pm_map = {
    'cash': 'Cash', 'efectivo': 'Cash',
    'credit card': 'Credit Card', 'tc': 'Credit Card', 'tarjeta de credito': 'Credit Card', 'tarjeta de crédito': 'Credit Card',
    'debit card': 'Debit Card', 'td': 'Debit Card', 'tarjeta de debito': 'Debit Card', 'tarjeta de débito': 'Debit Card',
    'mobile wallet': 'Mobile Wallet', 'mp': 'Mobile Wallet', 'mercado pago': 'Mobile Wallet', 'mercadopago': 'Mobile Wallet', 'billetera virtual': 'Mobile Wallet'
}
clean_df['payment_method'] = clean_df['payment_method'].fillna('').str.strip().str.lower().map(pm_map)

# Columna corregida: 'customer_email'
clean_df['email_norm'] = clean_df['customer_email'].fillna('').str.strip().str.lower()
email_to_cid = customers_clean.dropna(subset=['email']).set_index('email')['customer_id'].to_dict()
clean_df['customer_id'] = clean_df['email_norm'].map(email_to_cid)

stores['store_code'] = stores['store_code'].str.strip().str.upper()
clean_df = clean_df.merge(stores[['store_code', 'store_name']], on='store_code', how='left')
clean_df['total'] = np.round(clean_df['quantity'] * clean_df['unit_price'] * (1.0 - clean_df['discount']), 2)

cols_txn = [
    'txn_id', 'txn_datetime', 'store_code', 'store_name', 'customer_id', 'sku',
    'product_name', 'category', 'quantity', 'unit_price', 'price_source', 'discount',
    'payment_method', 'total'
]
transactions_clean = clean_df[cols_txn].copy()

for col in ['unit_price', 'discount', 'total']:
    transactions_clean[col] = pd.to_numeric(transactions_clean[col]).map(lambda x: f"{x:.2f}")

# ==========================================
# 4. RESUMEN MENSUAL (monthly_sales.csv)
# ==========================================
clean_df['month'] = clean_df['dt_ar'].dt.strftime('%Y-%m')
monthly_sales = clean_df.groupby(['month', 'store_code'], as_index=False).agg(
    transactions=('txn_id', 'count'),
    units=('quantity', 'sum'),
    revenue=('total', 'sum')
)
monthly_sales['revenue'] = np.round(monthly_sales['revenue'], 2)
monthly_sales = monthly_sales.sort_values(['month', 'store_code']).reset_index(drop=True)

# ==========================================
# 5. EXPORTAR CSVs (utf-8-sig para Excel)
# ==========================================
customers_clean.to_csv('customers_clean.csv', index=False, encoding='utf-8-sig')
transactions_clean.to_csv('transactions_clean.csv', index=False, encoding='utf-8-sig')
rejected_df.to_csv('transactions_rejected.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv('monthly_sales.csv', index=False, encoding='utf-8-sig')

print("=== RESUMEN DE AUDITORÍA ===")
print(f"Clientes originales: {orig_cust_count} | Duplicados fusionados: {cust_dups_removed} | Clientes limpios: {len(customers_clean)}")
print(f"Transacciones originales: {orig_txns_count} | Duplicados eliminados: {txn_dups_removed} | Limpias: {len(transactions_clean)} | Rechazadas: {len(rejected_df)}")
print("\nDesglose de rechazos:")
print(rejected_df['reason'].value_counts())
print("\nClientes asociados en ventas limpias:", transactions_clean['customer_id'].notna().sum())
print("Precios completados desde catálogo:", (transactions_clean['price_source'] == 'catalog').sum())

=== RESUMEN DE AUDITORÍA ===
Clientes originales: 3076 | Duplicados fusionados: 576 | Clientes limpios: 2500
Transacciones originales: 52000 | Duplicados eliminados: 4000 | Limpias: 45348 | Rechazadas: 2652

Desglose de rechazos:
reason
unknown_sku         939
invalid_quantity    557
invalid_store       489
invalid_discount    482
future_date         185
Name: count, dtype: int64

Clientes asociados en ventas limpias: 42248
Precios completados desde catálogo: 886
